# Companion Lab 04 — Real Protein Foundation Model Embeddings & Mutation Scoring (ESM-2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/04-protein-foundation-models-esm2.ipynb)

In **Chapter 4.4**, we built a 24-number hand-coded physicochemical protein embedding. Here we load Meta AI's real **ESM-2** protein language model (`facebook/esm2_t6_8M_UR50D`, pretrained on hundreds of millions of natural protein sequences!) to:
1. Compute **genuine contextual protein embeddings** and see biological families cluster in PCA space.
2. Run **zero-shot variant-effect scoring** on the real human **sickle-cell hemoglobin (`HBB E6V`)** mutation — watching ESM-2 score wild-type `E` vs. sickle-cell `V` purely from evolutionary grammar!

In [ ]:
!pip install -q transformers torch scikit-learn matplotlib numpy

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, EsmForMaskedLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'facebook/esm2_t6_8M_UR50D'   # Tip: try 'facebook/esm2_t12_35M_UR50D' on GPU!
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name).to(device).eval()
print(f'Loaded {model_name} ({model.num_parameters():,} parameters) on {device}')

## 1. Compute real ESM-2 embeddings for authentic biological protein families

In [ ]:
proteins = {
    'HBB_human':  'MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAH',
    'HBB_sickle': 'MVHLTPVEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAH',
    'HBA_human':  'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVA',
    'INS_human':  'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVG',
    'INS_pig':    'MALWTRLLPLLALLALWAPAPAQAFVNQHLCGSHLVEALYLVCGERGFFYTPKARREAENPQAG',
    'GFP_wild':   'MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTF',
    'Collagen':   'GPMGPMGPRGPPGPAGAPGPQGFQGNPGEPGEPGVSGPMGPRGPPGPPGKNGDDGEAGKPGRPG',
}

def esm_embed(seq):
    inputs = tokenizer(seq, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.esm(**inputs).last_hidden_state[0]
    # Mean-pool over residue tokens (excluding start/end special tokens)
    return out[1:-1].mean(dim=0).cpu().numpy()

names = list(proteins)
V = np.array([esm_embed(proteins[k]) for k in names])
V_unit = V / np.linalg.norm(V, axis=1, keepdims=True)
sim = V_unit @ V_unit.T
coords = PCA(n_components=2, random_state=0).fit_transform(V)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
im = ax1.imshow(sim, cmap='viridis', vmin=0.5, vmax=1.0)
ax1.set_xticks(range(len(names)), names, rotation=40, ha='right')
ax1.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax1.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if sim[i,j] < 0.85 else 'black')
ax1.set_title('A. ESM-2 learned cosine similarity')
fig.colorbar(im, ax=ax1, shrink=0.8)

ax2.scatter(coords[:, 0], coords[:, 1], s=110, color='#0072B2', edgecolors='black')
for i, name in enumerate(names):
    ax2.text(coords[i, 0] + 0.03, coords[i, 1] + 0.02, name, fontsize=9)
ax2.set_title('B. 2D PCA of 320-dim ESM-2 embeddings'); ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Zero-shot mutation scoring: masking the sickle-cell position (`HBB` pos 7 = `E`)

Remember Exercise 4.1.3! We mask out the exact amino acid mutated in sickle-cell disease (index 6 in `MVHLTPEEKS...`, which is `E` in normal human hemoglobin and `V` in sickle-cell hemoglobin) and ask ESM-2: *based purely on evolutionary sequence grammar, how probable is wild-type `E` versus mutant `V`?*

In [ ]:
hbb_seq = proteins['HBB_human']
masked_seq = hbb_seq[:6] + tokenizer.mask_token + hbb_seq[7:]
inputs = tokenizer(masked_seq, return_tensors='pt').to(device)
mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()

with torch.no_grad():
    probs = F.softmax(model(**inputs).logits[0, mask_idx], dim=-1)

for aa in ['E', 'D', 'Q', 'A', 'V']:
    tok_id = tokenizer.convert_tokens_to_ids(aa)
    print(f'P(amino acid = {aa} at HBB sickle site) = {probs[tok_id].item():.4f}')

p_wt = probs[tokenizer.convert_tokens_to_ids('E')].item()
p_mut = probs[tokenizer.convert_tokens_to_ids('V')].item()
print(f'\nLog-likelihood ratio log(P_mut / P_wt) = {np.log(p_mut / p_wt):+.2f} (negative = evolutionarily disfavored!)')

## 3. Modify & Make (PRIMM challenges)

1. **Compare to our reversed-gibberish test (Exercise 4.4.1):** Compute `esm_embed(proteins['HBB_human'][::-1])` and check its cosine similarity to real `HBB_human`. Unlike our toy bag-of-letters vector (`1.00`), ESM-2 knows sequence order matters!
2. **Scale up:** Switch `model_name = 'facebook/esm2_t12_35M_UR50D'` (or `650M` on a T4 GPU) and see how much sharper the `E vs. V` zero-shot log-likelihood gap becomes!